# Module 01 -- Black-Scholes from Scratch

**Author: Djellal Djouad** -- CrossVol Research | [crossvol.com](https://crossvol.com) | ORCID [0009-0002-4911-1118](https://orcid.org/0009-0002-4911-1118)

This notebook builds the Black-Scholes pricing model from first principles.
No libraries that hide the math -- just NumPy and the formulas.

I have priced thousands of options on a trading desk. The model is wrong.
It is wrong in specific, predictable ways that you can learn to exploit
once you understand what it assumes and where those assumptions break.
But you have to understand it first.

---
*License: MIT with Educational Use Clause -- see LICENSE. Not trading advice.*


In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt


In [ ]:
# The formula itself. Nothing fancy.
# S = spot, K = strike, T = time to expiry (years), r = risk-free rate,
# sigma = implied volatility, q = dividend yield

def black_scholes_call(S, K, T, r, sigma, q=0):
    """European call price under Black-Scholes."""
    if T <= 0:
        return max(S * np.exp(-q * T) - K, 0)
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


def black_scholes_put(S, K, T, r, sigma, q=0):
    """European put price under Black-Scholes."""
    if T <= 0:
        return max(K - S * np.exp(-q * T), 0)
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * np.exp(-q * T) * norm.cdf(-d1)


In [ ]:
# Quick sanity check -- SPX-style option
S = 5500      # spot
K = 5500      # ATM strike
T = 30 / 365  # 30 days
r = 0.05      # risk-free
sigma = 0.18  # 18% implied vol

call_price = black_scholes_call(S, K, T, r, sigma)
put_price = black_scholes_put(S, K, T, r, sigma)

print(f"Spot: {S}")
print(f"Strike: {K}")
print(f"Days to expiry: {T*365:.0f}")
print(f"Vol: {sigma:.0%}")
print(f"Call: ${call_price:.2f}")
print(f"Put:  ${put_price:.2f}")


## Put-Call Parity

This is not a model -- it is an arbitrage relationship. If it does not hold,
someone is about to lose money. On a desk, you check this before you check
anything else.


In [ ]:
# C - P = S*exp(-qT) - K*exp(-rT)
lhs = call_price - put_price
rhs = S * np.exp(0) - K * np.exp(-r * T)
print(f"C - P = {lhs:.4f}")
print(f"S - K*exp(-rT) = {rhs:.4f}")
print(f"Parity holds: {abs(lhs - rhs) < 0.001}")


## Payoff Diagrams

Every options conversation starts here. If you cannot draw the payoff
before you put the trade on, you should not put the trade on.


In [ ]:
spots = np.linspace(4800, 6200, 500)
premium_call = black_scholes_call(S, K, T, r, sigma)
premium_put = black_scholes_put(S, K, T, r, sigma)

# Payoff at expiry
call_payoff = np.maximum(spots - K, 0) - premium_call
put_payoff = np.maximum(K - spots, 0) - premium_put

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(spots, call_payoff, 'b-', linewidth=2)
axes[0].axhline(0, color='gray', linewidth=0.5)
axes[0].axvline(K, color='gray', linewidth=0.5, linestyle='--')
axes[0].set_title(f'Long Call -- K={K}, Premium=${premium_call:.2f}')
axes[0].set_xlabel('Spot at Expiry')
axes[0].set_ylabel('P&L')
axes[0].fill_between(spots, call_payoff, 0, where=call_payoff > 0, alpha=0.15, color='green')
axes[0].fill_between(spots, call_payoff, 0, where=call_payoff < 0, alpha=0.15, color='red')

axes[1].plot(spots, put_payoff, 'r-', linewidth=2)
axes[1].axhline(0, color='gray', linewidth=0.5)
axes[1].axvline(K, color='gray', linewidth=0.5, linestyle='--')
axes[1].set_title(f'Long Put -- K={K}, Premium=${premium_put:.2f}')
axes[1].set_xlabel('Spot at Expiry')
axes[1].set_ylabel('P&L')
axes[1].fill_between(spots, put_payoff, 0, where=put_payoff > 0, alpha=0.15, color='green')
axes[1].fill_between(spots, put_payoff, 0, where=put_payoff < 0, alpha=0.15, color='red')

plt.tight_layout()
plt.savefig('../data/01_payoff_diagrams.png', dpi=100, bbox_inches='tight')
plt.show()


## Price vs Spot -- Before Expiry

The option price is not the payoff. Before expiry, time value bends the curve.
This is where beginners get confused and where the money is.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for days in [90, 60, 30, 7, 1]:
    t = days / 365
    prices = [black_scholes_call(s, K, t, r, sigma) for s in spots]
    ax.plot(spots, prices, label=f'{days}d to expiry', linewidth=1.5)

# Intrinsic value
intrinsic = np.maximum(spots - K, 0)
ax.plot(spots, intrinsic, 'k--', label='Intrinsic (expiry)', linewidth=1)

ax.set_xlabel('Spot')
ax.set_ylabel('Call Price')
ax.set_title('Call Price vs Spot -- Time Value Decay')
ax.legend()
ax.set_xlim(4800, 6200)
plt.tight_layout()
plt.savefig('../data/01_time_value.png', dpi=100, bbox_inches='tight')
plt.show()


## Where Black-Scholes Breaks

The model assumes constant volatility, log-normal returns, no jumps,
continuous hedging, and zero transaction costs. None of these hold in
real markets. The volatility smile exists because the market knows
the model is wrong and prices the correction into every option.

Module 04 covers implied volatility. Module 09 covers stochastic vol (Heston).

---

**Next:** [Module 02 -- Greeks Intuition](02_greeks.ipynb)

**Further reading:** For a four-lens framework that goes beyond what
Black-Scholes can tell you about dealer positioning, see
[Beyond Gamma Exposure](https://www.amazon.com/dp/B0H2RZGMY6)
([working paper](https://doi.org/10.5281/zenodo.20509786)).

---
*Djellal Djouad -- CrossVol Research -- 2026*
